## DSPy Ollama Llama3 Information Extraction Pydantic

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch
from collections.abc import Iterable


from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer
from pydantic import BaseModel

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2', 'rougeL'], use_stemmer=True)
import json

from data.train_examples import train_example_list
from data.valid_examples import dev_example_list
from data.test_example import test_examples_list

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


#### Helper Functions

In [2]:
def flatten_list(nested_list):

    for item in nested_list:

        if isinstance(item, Iterable) and not isinstance(item, str):

            yield from flatten_list(item)

        else:

            yield item
            
def validate_ans(example, pred, trace = None):

    gold = re.sub(r'\n|\s+ ', '',dict(example)['info_extracted']).lower()
    print(gold)

    prediction = pred.info_extracted.lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Data

In [3]:
train_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet1.csv')

dev_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/50examples_for_DSPy_withJson.csv')

test_examples = pd.read_csv("/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet2.csv", header = None)

#### Set Ollama LLM using Llama 3 from Alibaba

In [4]:
llm = dspy.OllamaLocal(model='phi3:14b', max_tokens = 4000, temperature=0.0)
dspy.settings.configure(lm=llm)

#### Set Pydantic Class

In [5]:
class JobPostingExtraction(BaseModel):
    position_title: str
    location: str
    work_arrangement: str
    experience: str
    employment_type: str
    pay : str
    degree : str 
    certifications : str 
    required_skills : str

#### Create DSPy Signature 

In [6]:
class InfoExtractor(dspy.Signature):
    """Extract information from a job posting and return the output in a json format if you don't know answer Not Specified. Should be key-value with output as dictionary. """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted: JobPostingExtraction = dspy.OutputField(desc = "key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words all in json format")

#### Crease DSPy Module

In [7]:
class JobPostingModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.info_extraction = dspy.ChainOfThought(InfoExtractor)
    
    def forward(self, job_posting):
        job_posting = job_posting.replace('\n', ' ').replace('“', '"').replace('”', '"')
        job_posting = normalize(job_posting)
        info_extract = self.info_extraction(job_posting=job_posting).info_extracted.replace("```\n", "").replace("```", "")
        # dspy.Suggest(type(json.loads(dict(info_extract)['info_extracted'])) == dict, "Should return a dictionary format")
        # dspy.Suggest(list(json.loads(info_extract).keys()) == ['position_title','location','experience','employment_type','pay','degree','certifications','required_skills'],
        #              "The information extraction needs to have keys position_title','location','experience','employment_type','pay','degree','certifications','required_skills")
        return dspy.Prediction(job_posting= job_posting, info_extracted = info_extract)

In [8]:
uncompiled_module = JobPostingModule()

#### Perform Test Extraction on Data with uncompiled version

In [9]:
print(test_examples[1][5])

EMT-Advanced-Emergency Medical Service
Job Locations
US-TX-Rosenberg
Posted Date
9 months ago
(5/16/2023 11:01 AM)
ID 2023-5265 Job Code DJOB # of Openings 10 Min Start Salary USD $2,008.28/Bi. Category EMS Max Start Salary USD 
$2,421.41/Bi.
Overview

Fort Bend County is ranked as one of the fastest growing counties in the nation. We have capitalized on not only 
the creed of our location, but on the "quality of life" for our families to call home. Our employees are the key to
our success and the heartbeat of our foundation. The diversity and inclusivity of our community is our strength and
at the forefront of a workplace environment welcoming to all. Live Here! Work Here!



Provides emergency medical care to the citizens of Fort Bend County as stated in established standards and 
procedures.



Responsibilities
Provides emergency pre-hospital medical care. 
Completes reports within established timeframes.
Maintains emergency vehicle(s) and inventory of medical supplies.
Signs for and is held accountable for equipment issued and used.
Operates emergency vehicles (i.e. ambulance, squad) per department policy and with due regard to the law.
Responsible for maintaining all current certifications within department guidelines as required.
Certifications shall include EMT-Basic that has successfully completed AEMT and is test eligible and is enrolled in
an EMT Paramedic Program, DSHS EMT-Advanced Certification who is enrolled in an EMT Paramedic Program, and Valid 
State of Texas Driver's License.
Prepares, submits, and maintains clear, concise, and accurate documentation on patient care activities, incident 
reports and other related information as requested.
Assists other employees with their duties.
Performs general housekeeping duties for station and department areas.
Participates in activities and duties related to emergency management during a local state of disaster as directed 
by appropriate county managers.
Qualifications
High School Diploma/GED; Enrolled in College pursing Paramedic Certification and/or EMS Degree.
Certified or Licensed State of Texas EMT-Basic or EMT-Advanced or is eligible to test for EMT-Advanced 
Certification. 
Current Healthcare Provider CPR/AED card.
Pre-hospital experience preferred.
Experience in a high performance ALS system beneficial.
Strong verbal and written communication and organizational skills.
Strong interpersonal skills and ability to deal effectively with the public and other employees. 
Frequent reading, writing, memorization, analyzing, simple math skills, negotiating. Constantly using judgment, 
reasoning, decision-making and teaching.
Ability to complete projects.
Must obtain and maintain a current American Heart Association Advanced Cardiac Life Support certification.
Must complete National Incident Management System (NIMS) 100, 200, 700 and 800 within 90 days of hire.
Must obtain Paramedic Credentials in 24 months after hire date. Subject to emergency call-in and mandatory 
staffing.



SALARY RANGE: EMS Grade EMT-1, $2,008.28 - $2,421.41 biweekly based on qualifications

CLOSING DATE: Upon filling position





Fort Bend County is an equal opportunity employer, committed to non-discrimination in employment on any basis 
including race, color, religion or creed, sex, sexual orientation, gender, gender identity, gender expression, 
pregnancy status (including childbirth and related medical conditions), national origin, ethnicity, citizenship 
status, age (40 and over), physical or mental disability, genetic information, protected military and veteran 
status, political affiliation or beliefs, or any other classification protected by state, federal and local laws, 
unless such classification is a bona fide occupational qualification.

In [10]:
with dspy.context(lm = llm):
    pred = uncompiled_module(job_posting =test_examples[1][5])
    print(pred.info_extracted)

Job Title: Emergency Medical Services (EMS) Employee
Location: Fort Bend County, Texas
Key Responsibilities:
1. Prepare, submit, and maintain accurate documentation on patient care activities, incident reports, etc.
2. Assist other employees with their duties as needed.
3. Perform general housekeeping duties for station and department areas.
4. Participate in emergency management activities during a local state of disaster under the direction of county 
managers.
5. Obtain and maintain American Heart Association Advanced Cardiac Life Support certification.
6. Complete National Incident Management System (NIMS) 100, 200, 700, and 800 within ninety days of hire date.
7. Obtain paramedic credentials in 24 months after the hire date.

Qualifications:
- High school diploma/GED; enrolled in college pursuing EMT certification or an EMS degree.
- Certified or licensed Texas EMT-Basic, EMT-Advanced, or eligible to test for EMT-Advanced certification.
- Current healthcare provider CPR/AED card (preferred).
- Previous pre-hospital experience is beneficial.
- Experience in a high performance ALS system is advantageous.
- Strong verbal and written communication skills; organizational abilities.
- Ability to effectively interact with the public and other employees.
- Proficient in reading, writing, memorization, analysis, simple math, negotiation, judgment, reasoning, 
decision-making, and teaching.
- Capable of completing projects independently.

Salary Range: EMS Grade EMT-1 - $2,008.28 to $2,cuarterly based on qualifications
Closing Date: Upon filling the position
Fort Bend County is an equal opportunity employer committed to non-discrimination in employment.

#### Inspect History and save DSPy program 

In [11]:
#print(llm.inspect_history(n=1))
#uncompiled_module.save("uncompiled_file_pydantic.json")

#### Create Training Examples, Validation(Dev) and Test Examples

In [12]:
print(len(dev_example_list) , len(train_example_list), len(test_examples_list))
train_results = train_example_list
train_contents = list(train_examples['body'])

dev_results = dev_example_list
dev_contents = list(dev_examples.loc[:20,'body'])

test_results = test_examples_list
test_contents = list(test_examples[1])

21 20 10

In [13]:
train_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(train_contents, train_results)]
dev_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(dev_contents, dev_results)]
test_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(test_contents, test_results)]

In [14]:
trainset=train_examples_list
devset=dev_examples_list
testset = test_examples_list

trainset = [x.with_inputs('job_posting') for x in trainset]
devset = [x.with_inputs('job_posting') for x in devset]
testset = [x.with_inputs('job_posting') for x in testset]

#### Test Uncompiled Module on combined Rouge Score

In [15]:
answ = trainset[0]
print(answ)

Example({'job_posting': 'Derrickhand, Buckhannon, WV\n\n\nJob Order Number\n        \nWV2925359\n\n\nPost Date\n   
\n05/12/2023\n\n\nJob Location\n        \nBuckhannon, West Virginia 26201\n\n\nCounty\n        \nUpshur\n\n\nJob 
Summary\n        \nJob Summary: This position is a crew member assigned to work on a well service rig, responsible 
for performing services on oil and gas wells. Duties include performing all well-servicing tasks from an elevated 
position (rod basket or tubing board), assisting in rigging up or down, picking up or laying down tubing, and other
functions specified by the customer or well operator. This position has a dotted line reporting line to: Rig 
Supervisor. Responsibilities: Assists the operator in rigging up and down, lining up the well service rig with the 
well. Sets hydraulic jacks, handles pads/boards and assists in attaching the guy wires to the anchor. Responsible 
for all elevated work associated with rigging up/down (i.e. removing horse head from pumping unit). Responsible for
all work performed for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the
elevator), performs servicing on the well. Drives the crew truck as needed. Operates tubing elevators for standing 
tubing in derrick. Assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the 
work floor or vice versa. Assists in walking the rods when laying down rods. Reports any safety hazards, accidents 
or maintenance issues to the rig supervisor. Ensures that work carried out is in compliance with company policies 
and procedures and according to safety regulations. May be required to work floors or operate the rig when needed. 
Performs other related duties as assigned. Preferred Qualifications: 1-2 years of Workover - Derrickhand experience
required. Ability to effectively communicate, both verbally and written. Ability to interact with others in a team 
environment. Ability to work in a fast-paced environment and handle multiple tasks at once. Basic problem solving 
and organizational skills. Excellent customer service skills, to provide world class value to customers CDL B 
license is required to drive rig. Must meet all qualifications defined in the Motor Vehicle Policy if required to 
drive. Ability to communicate verbally and in writing, in English, is preferred. Education Requirements: High 
school diploma, GED, or the equivalent is preferred. We are proud to offer a very competitive compensation and 
benefits package including: Medical Insurance Vision and Dental Insurance Life Insurance 401(k) Education 
assistance Short-Term Disability Paid time-off Request Priority Protected Veteran Referrals Equal Opportunity 
Employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender 
identity\n\n\nExperience\n        \n0 Months\n\n\nPay Rate\n        \n0 $ / Hour\n\n\nMaster Group\n        
\nConstruction and Extraction Occupations\n\n\nJob Type\n        \nRotary Drill Operators, Oil and Gas\n\n\nShift\n
\nDay Shift', 'info_extracted': '\n    {\n        "position_title": "Derrickhand",\n        "location": 
"Buckhannon, West Virginia 26201",\n        "work_arrangement": "On-Site, Shifts",\n        "experience": "1-2 
years of Derrickhand experience",\n        "employment_type": "Full-time",\n        "pay": "Not specified",\n      
"degree": "High school diploma/GED or equivalent",\n        "certification": "CDL B License",\n        
"required_skills": "Effective verbal/written communication in English, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service"\n    }\n    '}) (input_keys={'job_posting'})

In [16]:
with dspy.context(lm=llm):
    pred = uncompiled_module(trainset[0].job_posting)
    print(pred.info_extracted)

Info Extracted: {
  "position_title": "Derrickhand",
  "location": "Buckhannon, West Virginia 26201",
  "work_arrangement": "On-site at well service rig",
  "experience": "1-2 years of workover - Derrickhand experience required",
  "employment_type": "Full-time",
  "pay": "$X/hour (not specified)",
  "degree": "High school diploma, GED, or equivalent preferred",
  "certifications": "CDL B license required for driving rig",
  "required_skills": [
    "Effective communication",
    "Teamwork",
    "Fast-paced environment handling",
    "Basic problem solving and organizational skills",
    "Excellent customer service"
  ]
}

In [17]:
validate_ans(answ, pred)

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

info extracted: {
  "position_title": "derrickhand",
  "location": "buckhannon, west virginia 26201",
  "work_arrangement": "on-site at well service rig",
  "experience": "1-2 years of workover - derrickhand experience required",
  "employment_type": "full-time",
  "pay": "$x/hour (not specified)",
  "degree": "high school diploma, ged, or equivalent preferred",
  "certifications": "cdl b license required for driving rig",
  "required_skills": [
    "effective communication",
    "teamwork",
    "fast-paced environment handling",
    "basic problem solving and organizational skills",
    "excellent customer service"
  ]
}

0.702608695652174

0.702608695652174

#### BootStrap Few Shot With A Few Examples to Change the Output

In [18]:
teleprompter = BootstrapFewShot(metric=validate_ans) 
compiled = teleprompter.compile(uncompiled_module, trainset=trainset)

  0%|          | 0/20 [00:00<?, ?it/s]

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

job title: derrickhand (well service rig crew member)
location: buckhannon, west virginia 26201 - county upshur
job post date: may cu2023
reporting line: dotted line reporting to the rig supervisor.
responsibilities:
- assist in rigging up and down, lining up well service rig with the well.
- set hydraulic jacks, handle pads/boards, and assist in attaching guy wires to anchor points.
- perform all elevated work associated with rigging up/down (e.g., walking rods, laying down tubing).
- report safety hazards, accidents, or maintenance issues to the rig supervisor.
- ensure compliance with company policies and procedures as well as safety regulations.
- may be required to work floors or operate the rig when needed.
- perform other related duties as assigned by supervisors.
preferred qualifications:
- 1-2 years of experience in a derrickhand role on a workover rig.
- ability to effectively communicate, both verbally and written.
- capability to interact with others in a team environment.
- able to work in fast-paced environments and handle multiple tasks simultaneously.
- basic problem solving and organizational skills.
- excellent customer service skills for providing world-class value to customers.
education requirements: high school diploma, ged, or equivalent is preferred.
additional qualifications: cdl b license required if driving the rig is part of job responsibilities; must meet all
qualifications defined in the motor vehicle policy (if applicable).
compensation and benefits package: competitive pay rate ($/hour), medical, vision, and dental insurance coverage, 
life insurance, 401(k) retirement plan with company match, education assistance program, short-term disability 
benefits, paid time off (pto).
equal opportunity employer.

0.14123847167325426

  5%|▌         | 1/20 [01:21<25:43, 81.23s/it]

...

 15%|█▌        | 3/20 [03:44<22:20, 78.86s/it]

{"position_title": "graphic designer","location": "goochland, va","work_arrangement": "hybrid, with two in-office 
days per week","experience": "minimum 5 years design and publications experience","employment_type": 
"part-time","pay": "not specified","degree": "college degree in graphic design, visual arts, or related 
field","certification": "not specified","required_skills": "proficiency with indesign, photoshop, illustrator, 
working knowledge of constant contact, strong organizational skills, excellent oral/written communication and 
client-relations skills, ability to work under pressure, working knowledge of ap style, 35mm and digital 
photography skills, mac environment"}

position title: graphic designer (part-time)
location: goochland, va
work arrangement: on-site, part-time
experience: minimum amoins 5 years of design and publications experience, preferably with a creative or marketing 
agency
employment type: part-time
pay: not specified
degree: college degree in graphic design, visual arts, or related field
certification: high proficiency in indesign, photoshop, illustrator; working knowledge of constant contact; minimum
skill qualifications include experience with publications and a strong portfolio of work
required skills: 
1. strong organizational skills to handle multiple projects
2. excellent oral and written communication and client-relations skills
3. ability to work effectively under pressure
4. working knowledge of ap style
5. 35mm and digital photography skills
6. mac environment preferred
7. direct mail experience and knowledge of postal standards
8. attention to detail
9. positive attitude, team player
10. high level of initiative and self-motivation
11. desire to continually grow one's skill set with ongoing education and training
benefits: 
1. exceptional benefits package including ongoing job development and support in all roles
2. paid training and continuing education reimbursement
3. medical and dental insurance available from the first day of employment
4. generous paid time off (pto) plan
company values: 
1. equal employment opportunity in all aspects of employment without regard to race, color, national origin, 
religion, gender, pregnancy, age, disability, orientation or veteran status
2. supports compliance with covid-nineteen protocols

0.30075570980743394

 20%|██        | 4/20 [05:35<22:20, 83.80s/it]


#### Test Compiled Version
    * Note: Some reason the compiled version works worst in this scenario, it might be due to the prompt or the metric being chosen that is affecting the output

In [19]:
with dspy.context(lm=llm):
    pred = compiled(job_posting=trainset[0].job_posting)
    print(pred.info_extracted)

Job Title: Crew Member (Derrickhand)
Location: Buckhannon, West Virginia
Responsibilities:
1. Assist the operator in rigging up and down, lining up the well service rig with the well.
2. Set hydralete jacks, handle pads, and report any safety hazards or maintenance issues to the rig supervisor.
3. Ensure work carried out is compliant with company policies, procedures, and safety regulations.
4. May be required to operate floors or the rig when needed.
5. Perform other related duties as assigned by the rig supervisor.
Preferred Qualifications:
1. 1-2 years of workover derrickhand experience
2. Effective communication skills (verbal and written) in English
3. Ability to interact with others in a team environment
4. Capability to work in a fast-paced environment, handling multiple tasks simultaneously
5. Basic problem-solving and organizational skills
6. Excellent customer service skills
7. CDL B license required for driving the rig
8. Ability to meet all qualifications defined in the motor vehicle policy if required to drive
Education Requirements: High school diploma, GED, or equivalent is preferred
Compensation and Benefits Package:
1. Medical insurance (including vision and dental)
2. Life insurance
3. Education assistance program
4. Short-term disability coverage
5. Paid time off
6. Priority protected veteran referrals
7. Equal opportunity employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender
identity

In [20]:
validate_ans(answ, pred)

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

job title: crew member (derrickhand)
location: buckhannon, west virginia
responsibilities:
1. assist the operator in rigging up and down, lining up the well service rig with the well.
2. set hydralete jacks, handle pads, and report any safety hazards or maintenance issues to the rig supervisor.
3. ensure work carried out is compliant with company policies, procedures, and safety regulations.
4. may be required to operate floors or the rig when needed.
5. perform other related duties as assigned by the rig supervisor.
preferred qualifications:
1. 1-2 years of workover derrickhand experience
2. effective communication skills (verbal and written) in english
3. ability to interact with others in a team environment
4. capability to work in a fast-paced environment, handling multiple tasks simultaneously
5. basic problem-solving and organizational skills
6. excellent customer service skills
7. cdl b license required for driving the rig
8. ability to meet all qualifications defined in the motor vehicle policy if required to drive
education requirements: high school diploma, ged, or equivalent is preferred
compensation and benefits package:
1. medical insurance (including vision and dental)
2. life insurance
3. education assistance program
4. short-term disability coverage
5. paid time off
6. priority protected veteran referrals
7. equal opportunity employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender
identity

0.17496089290999028

0.17496089290999028

In [21]:
compiled.save("compiled_v2_pydantic.json")

#### Do Side by Side Comparieson on Evaluation versus uncompiled vs compiled

In [22]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10,return_outputs=True)

prev_score=evaluation(uncompiled_module, metric=validate_ans)

improved_score=evaluation(compiled, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certifications": "crm (salesforce.com), rfq experience and price quotes to the dod",
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{
  "position_title": "federal sales engineer - tech & isr",
  "location": "walpole, ma, usa",
  "work_arrangement": "hybrid remote with 10-20% travel",
  "experience": "experience in power electronics or similar hardware development and sales to dod, homeland 
security, prime contractors, commercial industries",
  "employment_type": "full time",
  "pay": "$120k/year",
  "degree": "bs in engineering field",
  "certifications": "crm (salesforce.com) preferred, erp (epicor) familiarity preferred",
  "required_skills": "inside/outside technical sales experience, working with outside sales reps, rfq experience 
and price quotes to dod"
}

0.5197469197469198

Average Metric: 0.5197469197469198 / 1  (52.0):  10%|█         | 1/10 [00:31<04:45, 31.75s/it]

...

{"position_title": "cryptocurrency / fx trader - entry level", "location": "not specified", "work_arrangement": 
"remote", "experience": "no prior experience required", "employment_type": "full-time or part-time", "pay": 
"results-based commissions and performance bonuses", "degree": "bachelor's degree in finance, economics, or related
field preferred", "certifications": "none specified", "required_skills": "strong analytical skills, quick 
decision-making"}

job title: forex/cryptocurrency trader at maverick currencies

key responsibilities and skills required:
1. analyzing market data, news, and trends quickly and accurcuately
2. quick decision-making in a fast-paced environment
3. working efficiently in a mentally challenging environment
4. bachelor's degree preferred (finance, economics, or related field) but not required
5. strong understanding of financial markets, including forex, cryptocurrency, stocks, futures, options, etc.
6. proficient in using trading platforms and tools to execute trades effectively
7. ability to manage risk and maintain a disciplined approach to trading
8. continuous learning and staying updated with market developments

compensation:
- potential earnings of over $100,000 annually, with the possibility of unlimited earnings based on performance
- traders start with a minimum account size of $10,000 and have buying power up to $500,000
- retain 70-80% of trading profits
- eligibility for greater capital access and performance bonuses as experience and profitability increase

benefits:
1. flexible working hours - work full-time or part-nime from anywhere in the world
2. no recruitment agency involvement - direct application process
3. series 7, series 56, series 57, and series 65 certifications are advantageous for candidates with the necessary 
experience
4. math and statistics skills essential for analyzing market data and making informed trading decisions
5. analytical skills to interpret complex financial information and identify trends
6. communication skills - effective communication is crucial when working with clients, colleagues, or superiors
7. adaptability - capable of adjusting strategies based on market conditions and performance results
8. self-motivated - drive to succeed in a competitive environment without direct oversight

0.07677897692375187

Average Metric: 2.149261091090824 / 10  (21.5): 100%|██████████| 10/10 [07:57<00:00, 47.73s/it]


,example_job_posting,example_info_extracted,pred_job_posting,pred_info_extracted,validate_ans
0,"Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA Advanced Recruiting Solutions Walpole, MA Depends on Experience Full Time Work...","{""position_title"": ""Senior Inside Sales Rep/Sales Engineer"", ""location"": ""Walpole, MA"", ""work_arrangement"": ""Hybrid"", ""experience"": ""Depends on Experience"", ""employment_type"": ""Full Time"", ""pay"": ""$120K/year"", ""degree"": ""A BS in the...","federal sales engineer - tech & isr experience - hybrid remote - walpole ma advanced recruiting solutions walpole, ma depends on experience full time work...","Job Title: Federal Sales Engineer - Tech & ISR Location: Walpole, MA (within commuting distance) Employment Type: Full-time Compensation Range: $120k/year Degree Required: Bachelor's degree...",✔️ [0.19034743722050224]
1,"Position Description Penetration Tester Location Washington, DC Req # 12763 # of openings 2 ECS is seeking a Penetration Tester to work in our Washington,...","{""position_title"": ""Penetration Tester"", ""location"": ""Washington, DC"", ""work_arrangement"": ""On-site"", ""experience"": ""10+ years of Penetration Testing experience"", ""employment_type"": ""Full time"", ""pay"": ""Not specified"", ""degree"": ""Bachelors Degree in...","position description penetration tester location washington, dc req # 12763 # of openings 2 ecs is seeking a penetration tester to work in our washington,...","Job Title: Penetration Tester Location: Washington, DC Required Skills: 1. Bachelor's degree in Computer Science 2. 10+ years of penetration testing experience 3. Offensive security...",✔️ [0.1578410990175696]
2,NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility Inc. compensation: HOURLY...,"{""position_title"": ""NURSES - RNs or LPNs"", ""location"": ""Sudbury, MA 01776"", ""work_arrangement"": ""on-site"", ""experience"": ""Minimum of 1 year Long term care experience/SNF experience preferred"", ""employment_type"": ""full-time"",...",nurses - rns & lpns in snf - sign on bonus - child daycare on site (sudbury) sudbury pines extended care facility inc. compensation: hourly...,"Job Title: Nurse (RN/LPN) Company: Sudbury Pines Extended Care Facility Inc. Location: Sudbury, MA 01776 Employment Type: Full-time or Part-time Compensation: Hourly - Every other...",✔️ [0.17141302376201706]
3,Planner IV - Transportation Planner Job Details Apply Print Share This listing closes on 7/17/2023 at 11:59 PM Pacific Time (US & Canada); Tijuana. Salary...,"{""position_title"": ""Planner IV - Transportation Planner"", ""location"": ""Yakima, WA, 98901"", ""work_arrangement"": ""On-site"", ""experience"": ""5 years of increasingly responsible professional experience"", ""employment_type"": ""Full-Time"", ""pay"": ""$39.84 -...",planner iv - transportation planner job details apply print share this listing closes on 7/17/2023 at 11:59 pm pacific time (us & canada); tijuana. salary...,"Job Title: Planner IV - Transportation Planner Location: Yakima, WA Job Type: Full-time Department: Public Services - County Roads Closing Date: July 7, 2023 (11:59...",✔️ [0.37357330992098337]
4,"Associate Attorney Juan Ramos Law Group, PLLC McAllen, TX Job Details Full-time From $50,000 a year 1 day ago Qualifications Spanish Law Doctoral degree Criminal...","{""position_title"": ""Associate Attorney"", ""location"": ""McAllen, TX"", ""work_arrangement"": ""on-site"", ""experience"": ""None specified."", ""employment_type"": ""full-time"", ""pay"": ""$50,000 a year"", ""degree"": ""Law doctoral degree"", ""certifications"": ""Admission to the...","associate attorney juan ramos law group, pllc mcallen, tx job details full-time from $50,000 a year 1 day ago qualifications spanish law doctoral degree criminal...","Job Title: Associate Attorney Company: Juan Ramos Law Group, PLLC Location: Mcallen, TX Employment Type: Full-time P